# 🤝 Multi-Agent Systems: The Supervisor Pattern

## Learning Objectives
In this notebook, you will learn:
1. **Supervisor Architecture** - How a central "supervisor" node can coordinate multiple specialist agents in a LangGraph graph
2. **Structured Routing Decisions** - How to use a Pydantic schema with `with_structured_output` to get reliable routing choices from an LLM
3. **Specialist Agent Nodes** - How to design focused nodes (researcher, writer, critic) that each do one job well
4. **Conditional Routing** - How to wire `add_conditional_edges` so control flows back to the supervisor after each specialist runs
5. **Graph Assembly** - How to compile the individual pieces into a runnable multi-agent `StateGraph`

## Prerequisites
- Familiarity with LangGraph fundamentals (`StateGraph`, nodes, edges, conditional routing)
- Understanding of LangChain message types (`HumanMessage`, `AIMessage`, `SystemMessage`)
- An `OPENAI_API_KEY` configured in a `.env` file at the project root

---
## 📦 Setup

We import the LangGraph and LangChain building blocks we need, plus `pydantic` for the routing schema, and load environment variables from `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and API Keys
# ============================================================================
import operator
from typing import Literal

from typing_extensions import Annotated, TypedDict

from pydantic import BaseModel, Field

from dotenv import load_dotenv

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# Load API keys (OPENAI_API_KEY) from the .env file
load_dotenv()

print("✅ Imports loaded and environment variables configured!")

---
## 🗂️ Part 1: Defining the Shared State

Every LangGraph graph needs a state schema that all nodes read from and write to. For a supervisor-style multi-agent system, the state needs to track the running conversation, which agent should act next, and whether the task is complete.

### Key Concepts:
- **`messages`**: The shared conversation history, accumulated with `operator.add` so each node can append without overwriting prior turns
- **`next_agent`**: Set by the supervisor to tell the graph which specialist to route to next
- **`task_complete` / `final_response`**: Set once the supervisor decides the work is done

In [ ]:
# ============================================================================
# SUPERVISOR STATE: Shared Graph State Schema
# ============================================================================
class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # conversation history
    next_agent: str  # the agent that should act next
    task_complete: bool  # whether the task is complete or not
    final_response: str  # the final response to the user when the task is complete

### 🤖 Initializing the LLM

All nodes in this graph share a single LLM instance.

In [ ]:
# ============================================================================
# LLM INITIALIZATION
# ============================================================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

print(f"🤖 LLM initialized: {llm.model_name}")

---
## 🧭 Part 2: Structured Routing Decisions

The supervisor needs to make a reliable choice between four options: `researcher`, `writer`, `critic`, or `FINISH`. Rather than parsing free-form text, we define a Pydantic schema and bind it to the LLM with `with_structured_output`, so routing decisions come back as validated Python objects.

### `RouteDecision`
The Supervisor's routing decision, expressed as a typed Pydantic model.

In [ ]:
# ============================================================================
# ROUTE DECISION: Structured Output Schema for the Supervisor
# ============================================================================
# Define what the supervisor can decide
class RouteDecision(BaseModel):
    """The Supervisor's routing decision."""

    next: Literal["researcher", "writer", "critic", "FINISH"] = Field(
        description="Which agent to call next, or FINISH if done"
    )
    reasoning: str = Field(description="Why this agent was chosen")


# Create structured output for reliable routing
supervisor_llm = llm.with_structured_output(RouteDecision)

print("✅ Supervisor LLM configured with structured routing output!")

---
## 🧑‍💼 Part 3: The Supervisor Node

The `supervisor` node inspects the conversation so far and decides which specialist should act next, or whether the task is complete.

In [ ]:
# ============================================================================
# SUPERVISOR NODE: Decide Which Agent Acts Next
# ============================================================================
def supervisor(state: SupervisorState) -> dict:
    """The Supervisor decides what to do next."""
    system_prompt = """You are a supervisor managing a team of specialists:
    1. researcher - Gathers information and facts
    2. writer - Creates content and text
    3. critic - Reviews and improves work

    Based on the conversation, decide which agent should act next.

    Workflow:
    - Start with researcher to gather facts
    - Then writer to create content
    - Then critic to review
    - If critic suggests changes, send back to writer
    - When quality is good, respond with FINISH
    """

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    decision = supervisor_llm.invoke(messages)

    if decision.next == "FINISH":
        return {"next_agent": "FINISH", "task_complete": True}

    return {
        "next_agent": decision.next,
        "messages": [
            AIMessage(
                content=f"[Supervisor] Routing to {decision.next}: {decision.reasoning}"
            )
        ],
    }

---
## 👥 Part 4: Specialist Agent Nodes

Each specialist is a focused node with a single responsibility: gather facts, write content, or critique it. All three share the same LLM but use different system prompts.

### 🔬 `researcher`
Gathers information and facts about the original task.

In [ ]:
# ============================================================================
# RESEARCHER NODE: Gathers Information and Facts
# ============================================================================
def researcher(state: SupervisorState) -> dict:
    """Gathers information and facts."""
    system = """You are a research specialist. Your job:
    - Gather relevant facts and information
    - Be thorough but concise
    - Cite sources when possible
    - Focus on what's most useful for the task"""

    # Get the original task from the first human message
    task = next(
        (m.content for m in state["messages"] if isinstance(m, HumanMessage)), ""
    )

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Research this topic: {task}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Researcher] {response.content}")]}

### ✍️ `writer`
Creates content based on the research (and any critic feedback) available so far.

In [ ]:
# ============================================================================
# WRITER NODE: Creates Content Based on Available Information
# ============================================================================
def writer(state: SupervisorState) -> dict:
    """Creates content based on available information."""
    system = """You are a writing specialist. Your job:
    - Create clear, engaging content
    - Use the research provided
    - Match the requested format and tone
    - If there's critic feedback, incorporate it"""

    # Get recent context (research + any feedback)
    context = "\n".join([m.content for m in state["messages"][-5:]])

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Create content based on:\n{context}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Writer] {response.content}")]}

### 🧐 `critic`
Reviews the most recent work and either approves it or gives actionable feedback.

In [ ]:
# ============================================================================
# CRITIC NODE: Reviews Work and Provides Feedback
# ============================================================================
def critic(state: SupervisorState) -> dict:
    """Reviews work and provides feedback."""
    system = """You are a quality critic. Your job:
    - Review the content objectively
    - Provide specific, actionable feedback
    - If the work is good, say "APPROVED" and explain why
    - If it needs work, explain exactly what to improve"""

    # Get the most recent work
    context = "\n".join([m.content for m in state["messages"][-3:]])

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Review this work:\n{context}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Critic] {response.content}")]}

---
## 🏁 Part 5: Finalizing and Routing Logic

Two small helper functions close the loop: one extracts the final answer once the supervisor marks the task complete, and one translates the supervisor's decision into a graph edge target.

### `finalize`
Extracts the final response (the last Writer output) once the task is complete.

In [ ]:
# ============================================================================
# FINALIZE NODE: Extract the Final Response
# ============================================================================
def finalize(state: SupervisorState) -> dict:
    """Extract the final response when task is complete."""
    # Find the last Writer output
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage) and "[Writer]" in msg.content:
            content = msg.content.replace("[Writer] ", "")
            return {"final_response": content}
    return {"final_response": "Task completed."}

### `route_to_agent`
Routes the graph based on the Supervisor's decision - to a specialist, or to `finalize` when the task is done.

In [ ]:
# ============================================================================
# ROUTING FUNCTION: Route Based on Supervisor's Decision
# ============================================================================
def route_to_agent(state: SupervisorState) -> str:
    """Route based on Supervisor's decision."""
    if state.get("task_complete"):
        return "finalize"
    return state["next_agent"]

---
## 🕸️ Part 6: Assembling the Multi-Agent Graph

With every node and the routing function defined, we can wire them into a `StateGraph`: the supervisor always runs first, routes to a specialist via conditional edges, and every specialist routes back to the supervisor until it decides to `FINISH`.

### `build_multi_agent_system`
Builds and compiles the complete multi-agent graph.

In [ ]:
# ============================================================================
# GRAPH ASSEMBLY: Build the Complete Multi-Agent System
# ============================================================================
def build_multi_agent_system():
    """Build the complete multi-agent graph."""
    # Create graph
    graph = StateGraph(SupervisorState)

    # Add all nodes
    graph.add_node("supervisor", supervisor)
    graph.add_node("researcher", researcher)
    graph.add_node("writer", writer)
    graph.add_node("critic", critic)
    graph.add_node("finalize", finalize)

    # Entry point: always start with Supervisor
    graph.add_edge(START, "supervisor")

    # Supervisor routes to specialists
    graph.add_conditional_edges(
        "supervisor",
        route_to_agent,
        {
            "researcher": "researcher",
            "writer": "writer",
            "critic": "critic",
            "finalize": "finalize",
        },
    )

    # After each specialist, go back to Supervisor
    graph.add_edge("researcher", "supervisor")
    graph.add_edge("writer", "supervisor")
    graph.add_edge("critic", "supervisor")

    # Finalize ends the graph
    graph.add_edge("finalize", END)

    return graph.compile()


print("✅ build_multi_agent_system() defined!")

---
## ▶️ Part 7: Running the Multi-Agent System

This is the original `__main__` guard, kept as-is: since Jupyter sets `__name__` to `"__main__"`, the cell runs automatically. It builds the graph, seeds an initial task, and prints the full agent conversation followed by the final output.

In [ ]:
# ============================================================================
# RUN: Build the Graph and Execute a Sample Task
# ============================================================================
if __name__ == "__main__":
    # Build the system
    agent = build_multi_agent_system()

    # Initial state
    initial_state = {
        "messages": [
            HumanMessage(
                content="Write a short blog post about the benefits of AI in healthcare"
            )
        ],
        "next_agent": "",
        "task_complete": False,
        "final_response": "",
    }

    print("=" * 60)
    print("MULTI-AGENT SYSTEM")
    print("=" * 60)

    # Run the system
    result = agent.invoke(initial_state)

    # Show the conversation
    print("\nAgent Conversation:")
    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            # Truncate for display
            content = (
                msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
            )
            print(f"\n{content}")

    print("\n" + "=" * 60)
    print("FINAL OUTPUT:")
    print("=" * 60)
    print(result["final_response"])

---
## 📝 Summary

In this notebook, we built a supervisor-style multi-agent system with LangGraph.

### 1. State and Routing
- **`SupervisorState`**: Shared graph state tracking messages, the next agent, and task completion
- **`RouteDecision`**: A Pydantic schema used with `with_structured_output` for reliable supervisor routing
- **`route_to_agent`**: Translates the supervisor's decision into a graph edge target

### 2. Nodes
- **`supervisor`**: Decides which specialist should act next, or whether to finish
- **`researcher`**, **`writer`**, **`critic`**: Focused specialist nodes, each with a single responsibility
- **`finalize`**: Extracts the final response once the supervisor marks the task complete

### 3. Graph Assembly
- **`build_multi_agent_system`**: Wires all nodes together - supervisor routes out via conditional edges, and every specialist routes back to the supervisor until `FINISH`

### Next Steps
- Continue to **`02_supervisor_agent.ipynb`** to explore the supervisor pattern in more depth